# SkyPortal Corpus Structure

This notebook establishes the raw SkyPortal population used by the project before any
ledger, event matching, or model-facing transformation exists. It uses one frozen
capture, **2026-07-20**. SkyPortal is a moving target, so every count below is tied to
that date.

Each section follows the same sequence: **question, measurement, decision**.


## 1. Scope and capture

**QUESTION.** Which four inventory profiles define the source population in the frozen
capture?


In [1]:
from pathlib import Path
import json
import re
import sys

import pandas as pd
import yaml

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_ROOT = ROOT / "data" / "raw" / "skyportal" / "inventory"
EVIDENCE_DIR = ROOT / "notebooks" / "evidence"
CONFIG_PATH = ROOT / "configs" / "extraction" / "skyportal.yaml"
CAPTURE_STAMP = "20260720"
PROFILES = ["grandma_base", "gcn", "ep", "grb"]


def is_non_empty(value):
    return value is not None and value != "" and value != [] and value != {}


def classify_name(source_id):
    if re.fullmatch(r"GCN-\d{6}_\d{6}", source_id, re.IGNORECASE):
        return "gcn_internal"
    if re.fullmatch(r"EP-\d{6}_\d{6}", source_id, re.IGNORECASE):
        return "ep_internal"
    if re.fullmatch(r"GRB-\d{6}_\d{6}", source_id, re.IGNORECASE):
        return "grb_internal"
    if re.match(r"GRB", source_id, re.IGNORECASE):
        return "grb_named"
    if re.match(r"(AT|SN)\s?\d{4}", source_id, re.IGNORECASE):
        return "tns_like"
    if re.match(r"ZTF", source_id, re.IGNORECASE):
        return "ztf_like"
    return "other"


config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
profile_rows = []
records_by_profile = {}
records_by_source = {}

for profile_name in PROFILES:
    profile = config["inventory"]["profiles"][profile_name]
    template_name = profile["query_template"]
    query_params = dict(config["inventory"]["shared_query_params"][template_name])
    query_params.update(profile.get("query_params", {}))
    run_directories = sorted(
        RAW_ROOT.glob(f"source_inventory_{profile_name}_{CAPTURE_STAMP}_*")
    )
    if len(run_directories) != 1:
        raise RuntimeError(
            f"Expected one frozen run for {profile_name}, found {len(run_directories)}"
        )

    records = []
    for page_path in sorted(run_directories[0].glob("sources_page_*.json")):
        payload = json.loads(page_path.read_text(encoding="utf-8"))
        records.extend(
            row
            for row in payload.get("data", {}).get("sources", [])
            if isinstance(row, dict)
        )
    records_by_profile[profile_name] = records
    for record in records:
        source_id = record.get("id")
        if isinstance(source_id, str):
            records_by_source.setdefault(source_id, {})[profile_name] = record

    profile_rows.append(
        {
            "profile": profile_name,
            "query_template": template_name,
            "query_params": json.dumps(query_params, sort_keys=True),
            "raw_directory": run_directories[0].name,
        }
    )

july_rows = [record for rows in records_by_profile.values() for record in rows]
print(f"Python: {sys.executable}")
print("Frozen capture: 2026-07-20")
print(pd.DataFrame(profile_rows).to_string(index=False))


Python: /home/meneses/project_astronomical/MAFORAI/.venv/bin/python
Frozen capture: 2026-07-20
     profile       query_template                                                                                                                                                                                                               query_params                                 raw_directory
grandma_base enriched_recent_desc {"group_ids": "3", "includeCommentExists": "true", "includeDetectionStats": "true", "includeHosts": "true", "includePhotometryExists": "true", "includeSpectrumExists": "true", "sortBy": "saved_at", "sortOrder": "desc"} source_inventory_grandma_base_20260720_093939
         gcn enriched_recent_desc                        {"includeCommentExists": "true", "includeDetectionStats": "true", "includePhotometryExists": "true", "includeSpectrumExists": "true", "sortBy": "saved_at", "sortOrder": "desc", "sourceID": "GCN"}          source_inventory_gcn_20260720_093955
        

**DECISION.** The corpus population is the union of the `grandma_base`, `gcn`, `ep`,
and `grb` profiles printed above. Changing any query changes the population and creates
a different dated capture.


## 2. Population

**QUESTION.** How many records and unique sources are present, and how much overlap is
there between profiles?


In [2]:
population = pd.DataFrame(
    [
        {
            "profile": profile_name,
            "records": len(records_by_profile[profile_name]),
            "unique_source_ids": len(
                {record["id"] for record in records_by_profile[profile_name]}
            ),
        }
        for profile_name in PROFILES
    ]
)

selected_records = {}
source_index_rows = []
for source_id in sorted(records_by_source):
    profile_records = records_by_source[source_id]
    _, selected_record = max(
        profile_records.items(),
        key=lambda item: (
            sum(is_non_empty(value) for value in item[1].values()),
            item[0],
        ),
    )
    selected_records[source_id] = selected_record
    source_index_rows.append(
        {
            "source_id": source_id,
            "profiles": "|".join(sorted(profile_records)),
            "name_pattern_class": classify_name(source_id),
            "has_t0": is_non_empty(selected_record.get("t0")),
            "t0": selected_record.get("t0"),
            "n_redshift_versions": len(selected_record.get("redshift_history") or []),
            "n_summary_versions": len(selected_record.get("summary_history") or []),
            "n_classifications": len(selected_record.get("classifications") or []),
            "has_alias": is_non_empty(selected_record.get("alias")),
            "has_tns_name": is_non_empty(selected_record.get("tns_name")),
            "comment_exists": bool(selected_record.get("comment_exists")),
            "photometry_exists": bool(selected_record.get("photometry_exists")),
            "spectrum_exists": bool(selected_record.get("spectrum_exists")),
            "created_at": selected_record.get("created_at"),
            "modified": selected_record.get("modified"),
            "ra": selected_record.get("ra"),
            "dec": selected_record.get("dec"),
            "redshift": selected_record.get("redshift"),
        }
    )

source_index = pd.DataFrame(source_index_rows)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
source_index.to_csv(EVIDENCE_DIR / "01_source_index.csv", index=False)

n_profile_overlaps = source_index["profiles"].str.contains(r"\|").sum()
print(population.to_string(index=False))
print(f"All July listing records: {len(july_rows)}")
print(f"Unique July source IDs: {len(source_index)}")
print(f"Sources in two or more profiles: {n_profile_overlaps}")


     profile  records  unique_source_ids
grandma_base      407                407
         gcn      155                155
          ep      219                219
         grb      201                201
All July listing records: 982
Unique July source IDs: 800
Sources in two or more profiles: 182


**FINDING.** The working population contains **800 unique sources** represented by
**982 listing records**. The four profiles contain 407 `grandma_base`, 155 `gcn`, 219
`ep`, and 201 `grb` records; 182 sources occur in at least two profiles.


## 3. Profile consistency

**QUESTION.** When one source appears in multiple profiles, do its key set or
scientific history arrays change?


In [3]:
overlapping_ids = sorted(
    source_id
    for source_id, profile_records in records_by_source.items()
    if len(profile_records) > 1
)
key_differences = set()
value_differences = {
    field: 0
    for field in [
        "redshift_history",
        "summary_history",
        "classifications",
        "annotations",
    ]
}
examples = []

for source_id in overlapping_ids:
    profile_records = records_by_source[source_id]
    profile_names = sorted(profile_records)
    left_name, right_name = profile_names[:2]
    left = profile_records[left_name]
    right = profile_records[right_name]
    key_differences.update(set(left) ^ set(right))
    for field in value_differences:
        if left.get(field) != right.get(field):
            value_differences[field] += 1
    if len(examples) < 3:
        examples.append(
            {
                "source_id": source_id,
                "profiles": f"{left_name}|{right_name}",
                "key_difference": "|".join(sorted(set(left) ^ set(right))),
                **{
                    f"{field}_same": left.get(field) == right.get(field)
                    for field in value_differences
                },
            }
        )

print(f"Overlapping sources compared: {len(overlapping_ids)}")
print(f"Cross-profile key differences: {sorted(key_differences)}")
print(pd.Series(value_differences, name="different_sources").to_string())
print(pd.DataFrame(examples).to_string(index=False))


Overlapping sources compared: 182
Cross-profile key differences: ['host', 'host_offset']
redshift_history    0
summary_history     0
classifications     0
annotations         0
       source_id        profiles   key_difference  redshift_history_same  summary_history_same  classifications_same  annotations_same
EP-241103_012438 ep|grandma_base host|host_offset                   True                  True                  True              True
EP-250704_081653 ep|grandma_base host|host_offset                   True                  True                  True              True
EP-250727_073405 ep|grandma_base host|host_offset                   True                  True                  True              True


**DECISION.** Deduplicate each source ID by selecting the profile record with the most
non-empty fields; profile name is the deterministic tie-break. Only `host` and
`host_offset` differ by key presence, and the four scientific arrays compared above
are value-identical.


## 4. What the old compact file discarded

**QUESTION.** What did the historical GCN-derived name filter remove from the raw
800-source population, and did those sources carry data?


In [4]:
def extract_aliases(source):
    raw_aliases = source.get("alias")
    if isinstance(raw_aliases, str):
        raw_items = [raw_aliases]
    elif isinstance(raw_aliases, list):
        raw_items = raw_aliases
    else:
        return []
    return [
        item.strip()
        for item in raw_items
        if isinstance(item, str) and item.strip()
    ]


def classify_gcn_derived_type(source_id):
    source_id = source_id.strip()
    if source_id.startswith("GRB"):
        return "grb"
    if source_id.startswith("GW"):
        return "gw"
    if source_id.startswith("EP"):
        return "ep"
    if source_id.startswith("GCN"):
        return "gcn"
    return "other"


def classify_gcn_derived_source(source):
    source_id = str(source.get("id", "")).strip()
    source_type = classify_gcn_derived_type(source_id)
    if source_type != "other":
        return source_type
    for alias in extract_aliases(source):
        alias_type = classify_gcn_derived_type(alias)
        if alias_type != "other":
            return alias_type
    return "other"


filtering_code = '''def build_gcn_grandma_event(source: dict[str, Any]) -> dict[str, Any] | None:
    """Build one compact GCN-derived source record."""
    source_id = str(source.get("id", "")).strip()
    source_type = classify_gcn_derived_source(source)
    if source_type == "other":
        return None'''

compact_ids = {
    source_id
    for source_id, record in selected_records.items()
    if classify_gcn_derived_source(record) != "other"
}
excluded = source_index.loc[~source_index["source_id"].isin(compact_ids)].copy()
excluded_has_data = (
    excluded["comment_exists"]
    | excluded["photometry_exists"]
    | excluded["spectrum_exists"]
    | excluded["has_t0"]
    | (excluded["n_classifications"] > 0)
)

print(f"Raw population: {len(source_index)}")
print(f"Old compact-eligible population: {len(compact_ids)}")
print(f"Excluded by the historical filter: {len(excluded)}")
print(f"Excluded with a data-bearing flag: {int(excluded_has_data.sum())}")
print(excluded["name_pattern_class"].value_counts().rename_axis("name_pattern_class").to_string())
print("\nHistorical filtering fragment (verbatim):")
print(filtering_code)


Raw population: 800
Old compact-eligible population: 576
Excluded by the historical filter: 224
Excluded with a data-bearing flag: 196
name_pattern_class
other       186
ztf_like     25
tns_like     13

Historical filtering fragment (verbatim):
def build_gcn_grandma_event(source: dict[str, Any]) -> dict[str, Any] | None:
    """Build one compact GCN-derived source record."""
    source_id = str(source.get("id", "")).strip()
    source_type = classify_gcn_derived_source(source)
    if source_type == "other":
        return None


**DECISION.** The 224 sources rejected by the historical name filter remain part of
the corpus: 196 carry at least one checked data-bearing flag. The emitter must retain
the four-profile union and must not filter population membership by source name.


## 5. Field coverage

**QUESTION.** Which fields are always, partially, or never populated across the 982
raw listing records?


In [5]:
all_fields = sorted({field for record in july_rows for field in record})
field_coverage = pd.DataFrame(
    [
        {
            "field": field,
            "n_present": sum(field in record for record in july_rows),
            "n_non_empty": sum(
                field in record and is_non_empty(record[field])
                for record in july_rows
            ),
        }
        for field in all_fields
    ]
)
field_coverage["pct_non_empty"] = (
    100.0 * field_coverage["n_non_empty"] / len(july_rows)
)
field_coverage = field_coverage.sort_values(
    ["pct_non_empty", "field"], ascending=[False, True]
).reset_index(drop=True)
field_coverage.to_csv(EVIDENCE_DIR / "01_field_coverage.csv", index=False)

field_coverage["bucket"] = pd.cut(
    field_coverage["pct_non_empty"],
    bins=[-0.1, 0, 99.999999, 100],
    labels=["never", "partial", "always"],
)
bucket_summary = (
    field_coverage.groupby("bucket", observed=True)["field"]
    .agg(n_fields="size", fields=lambda values: ", ".join(values))
    .reset_index()
)
print(field_coverage[["field", "pct_non_empty"]].to_string(
    index=False, formatters={"pct_non_empty": lambda value: f"{value:.2f}%"}
))
print("\nCoverage buckets:")
print(bucket_summary.to_string(index=False))


                    field pct_non_empty
           comment_exists       100.00%
               created_at       100.00%
                      dec       100.00%
                  gal_lat       100.00%
                  gal_lon       100.00%
                   groups       100.00%
                  healpix       100.00%
                       id       100.00%
             internal_key       100.00%
                  is_roid       100.00%
                 modified       100.00%
                   offset       100.00%
        photometry_exists       100.00%
                       ra       100.00%
          spectrum_exists       100.00%
                transient       100.00%
                  varstar       100.00%
                   origin        45.93%
          summary_history        39.10%
                  summary        38.19%
                photstats        37.78%
          classifications        36.76%
                     tags        30.14%
                    alias        29.02%


**DECISION.** Preserve as fact inputs the source identity and timing fields `id`,
`alias`, `tns_name`, `created_at`, `modified`, and `t0`; coordinates and redshift
fields `ra`, `dec`, `redshift`, `redshift_error`, `redshift_origin`, and
`redshift_history`; and structured scientific histories `summary_history`,
`classifications`, `annotations`, `gcn_crossmatch`, `host`, `host_offset`, and
`photstats`. Keep other non-empty fields as raw context. Omit the empirically empty
fields from the structural extract rather than promoting them to fact types.


## 6. Is the listing truncated?

**QUESTION.** Do listing-level redshift and summary histories reproduce the counts
previously observed from the per-source endpoint?


In [6]:
controls = pd.DataFrame(
    [
        ("2025aji", 4, 8),
        ("GRB241030", 1, 6),
        ("2026owq", 2, 44),
        ("EP-260623_025405", 2, 24),
    ],
    columns=[
        "source_id",
        "expected_redshift_versions",
        "expected_summary_versions",
    ],
)
control_rows = controls.merge(
    source_index[["source_id", "n_redshift_versions", "n_summary_versions"]],
    on="source_id",
    how="left",
)
control_rows["redshift_matches"] = (
    control_rows["expected_redshift_versions"]
    == control_rows["n_redshift_versions"]
)
control_rows["summary_matches"] = (
    control_rows["expected_summary_versions"]
    == control_rows["n_summary_versions"]
)
print(control_rows.to_string(index=False))


       source_id  expected_redshift_versions  expected_summary_versions  n_redshift_versions  n_summary_versions  redshift_matches  summary_matches
         2025aji                           4                          8                    4                   8              True             True
       GRB241030                           1                          6                    1                   6              True             True
         2026owq                           2                         44                    2                  44              True             True
EP-260623_025405                           2                         24                    2                  24              True             True


**DECISION.** All four controls match: `2025aji` 4/8, `GRB241030` 1/6,
`2026owq` 2/44, and `EP-260623_025405` 2/24 redshift/summary versions. On
this evidence, source-level histories can be read from the listing; row-level detail
still requires per-source downloads.


## 7. What the listing does not contain

**QUESTION.** Are comments, photometry, spectra, and follow-up requests present as
rows, or only represented by availability flags?


In [7]:
coverage_fields = set(field_coverage["field"])
collections = ["comments", "photometry", "spectra", "followup_requests"]
collection_check = pd.DataFrame(
    {
        "collection": collections,
        "top_level_rows_present": [
            field in coverage_fields for field in collections
        ],
    }
)
flag_check = pd.DataFrame(
    {
        "flag": ["comment_exists", "photometry_exists", "spectrum_exists"],
        "meaning": [
            "availability flag, not comment rows",
            "availability flag, not photometry rows",
            "availability flag, not spectrum rows",
        ],
    }
)

print(collection_check.to_string(index=False))
print(flag_check.to_string(index=False))
print(f"Minimum collection requests: {len(source_index)} x 4 = {len(source_index) * 4}")


       collection  top_level_rows_present
         comments                   False
       photometry                   False
          spectra                   False
followup_requests                   False
             flag                                meaning
   comment_exists    availability flag, not comment rows
photometry_exists availability flag, not photometry rows
  spectrum_exists   availability flag, not spectrum rows
Minimum collection requests: 800 x 4 = 3200


**DECISION.** A per-source download stage is required for comments, photometry,
spectra, and follow-up requests. For 800 sources, the complete four-collection pass
starts at 3,200 requests and therefore requires pacing and checkpoint/resume.


## Decisions summary

The measurements above establish the structural boundary for the next notebook and
for the ingestion stages that follow.


In [8]:
decisions = pd.DataFrame(
    [
        ("Use the union of four profiles", "982 records / 800 sources", 1),
        ("Retain 800 unique sources", "182 cross-profile overlaps", 2),
        ("Deduplicate by richest record", "Only host and host_offset keys differ", 3),
        ("Do not filter by source name", "224 excluded; 196 data-bearing", 4),
        ("Separate fact inputs from raw context", "Measured field coverage", 5),
        ("Read source histories from listing", "Four controls match", 6),
        ("Download row-level collections per source", "Rows absent; flags present", 7),
    ],
    columns=["decision", "evidence", "section"],
)
print(decisions.to_string(index=False))


                                 decision                              evidence  section
           Use the union of four profiles             982 records / 800 sources        1
                Retain 800 unique sources            182 cross-profile overlaps        2
            Deduplicate by richest record Only host and host_offset keys differ        3
             Do not filter by source name        224 excluded; 196 data-bearing        4
    Separate fact inputs from raw context               Measured field coverage        5
       Read source histories from listing                   Four controls match        6
Download row-level collections per source            Rows absent; flags present        7


**DECISION.** These seven decisions define the raw corpus structure. Temporal-anchor
reconstruction is intentionally outside this notebook and begins from the structural
`01_source_index.csv` written here.
